In [2]:
from pathlib import Path
dvc_file = Path('data/Telco-Customer-Churn.csv.dvc')
dvc_metadata = dvc_file.read_text()
print(dvc_metadata)

outs:
- md5: 836fd67bbd5c4bad12630810a4cd0f48
  size: 970458
  isexec: true
  hash: md5
  path: Telco-Customer-Churn.csv



In [1]:
import mlflow
import mlflow.sklearn
import optuna
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from xgboost import XGBClassifier
from mlflow.models import infer_signature

data = pd.read_csv("data/Telco-Customer-Churn.csv")

data.drop("customerID",axis=1,inplace=True)

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"],errors="coerce")

x = data.drop("Churn",axis=1)
y = data["Churn"].map({"No":0,"Yes":1})

num = x.select_dtypes(include=["int64","float64"]).columns.tolist()
cat = x.select_dtypes(include=["object"]).columns.tolist()

numerical_pipeline = Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])

categorical_pipeline = Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore",drop="first"))])

preprocessing = ColumnTransformer([("num",numerical_pipeline,num),("cat",categorical_pipeline,cat)])

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

input_example = x_train.head()

mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("customer_churn_v2")

def objective(trial):

    params = {
        "max_depth":trial.suggest_int("max_depth",3,6),
        "n_estimators":trial.suggest_int("n_estimators",50,200),
        "learning_rate":trial.suggest_float("learning_rate",0.01,0.3,log=True),
        "subsample":trial.suggest_float("subsample",0.6,1.0),
        "colsample_bytree":trial.suggest_float("colsample_bytree",0.6,1.0)
    }

    with mlflow.start_run(nested=True):

        model = Pipeline([("preprocess",preprocessing),("model",XGBClassifier(**params))])

        model.fit(x_train,y_train)

        pred = model.predict(x_test)

        acc = accuracy_score(y_test,pred)

        mlflow.log_params(params)
        mlflow.log_param("dataset","customer_churn")
        mlflow.log_metric("accuracy",acc)

        return acc

with mlflow.start_run():

    study = optuna.create_study(direction="maximize")

    study.optimize(objective,n_trials=50)

    best_model = Pipeline([("preprocess",preprocessing),("model",XGBClassifier(**study.best_params))])

    best_model.fit(x_train,y_train)

    signature = infer_signature(x_train,best_model.predict(x_train))

    mlflow.sklearn.log_model(best_model,artifact_path="customer_model",input_example=input_example,signature=signature)

    mlflow.log_params(study.best_params)

    mlflow.log_metric("best_accuracy",study.best_value)

    mlflow.log_param('dataset_file','data/Telco-Customer-Churn.csv.')
    mlflow.log_param('dataset_version','3b0bfab28a8101b4e4fdd08025a5c235')
    final_pred = best_model.predict(x_test)
    final_accuracy = accuracy_score(y_test, final_pred)
    mlflow.log_metric("final_model_accuracy", final_accuracy)
    print(study.best_params)
    print(study.best_value)

2026/10/28 20:24:26 INFO mlflow.tracking.fluent: Experiment with name 'customer_churn_v2' does not exist. Creating a new experiment.
[I 2026-10-28 20:24:27,712] A new study created in memory with name: no-name-1f93772d-ab37-44df-981c-35e91e0d2e42
[I 2026-10-28 20:24:29,390] Trial 0 finished with value: 0.8069552874378992 and parameters: {'max_depth': 3, 'n_estimators': 180, 'learning_rate': 0.142964058278682, 'subsample': 0.7894102822862634, 'colsample_bytree': 0.7826414156677504}. Best is trial 0 with value: 0.8069552874378992.


🏃 View run amusing-ant-71 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/e5d7e8fe0f8b471e89c598048022b485
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


[I 2026-10-28 20:24:30,312] Trial 1 finished with value: 0.8090844570617459 and parameters: {'max_depth': 4, 'n_estimators': 164, 'learning_rate': 0.09534812721454046, 'subsample': 0.8738835167310037, 'colsample_bytree': 0.7291125384034132}. Best is trial 1 with value: 0.8090844570617459.


🏃 View run orderly-conch-853 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/c80b741227c24f348b5489ec169000d0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


[I 2026-10-28 20:24:31,128] Trial 2 finished with value: 0.808374733853797 and parameters: {'max_depth': 6, 'n_estimators': 147, 'learning_rate': 0.03238096129828196, 'subsample': 0.8656446490599519, 'colsample_bytree': 0.7909633382286261}. Best is trial 1 with value: 0.8090844570617459.


...

🏃 View run efficient-fox-34 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/c7709434e54440659475365f4d5c575e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


[I 2026-10-28 20:25:10,085] Trial 49 finished with value: 0.8190205819730305 and parameters: {'max_depth': 3, 'n_estimators': 139, 'learning_rate': 0.07262662536843144, 'subsample': 0.708838078007896, 'colsample_bytree': 0.9307691239542168}. Best is trial 31 with value: 0.8204400283889283.


🏃 View run bright-seal-503 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/f94282fa294c498d8f7ae22b650b6335
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


/mnt/d/brototype/SAFAN/code/fourtysix/mlops/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/28 20:25:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'max_depth': 3, 'n_estimators': 160, 'learning_rate': 0.060667474619279106, 'subsample': 0.6543429272038939, 'colsample_bytree': 0.8546478695943032}
0.8204400283889283
🏃 View run magnificent-eel-944 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/b06f26560db045b1906c6beb75b60287
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


In [9]:
from pathlib import Path

csv_path = Path("data/Telco-Customer-Churn.csv")

with csv_path.open('ab') as file:
    file.write(b'\n')

In [ ]:
# git init
# dvc init
# dvc add data/Telco-Customer-Churn.csv
# git add data/Telco-Customer-Churn.csv.dvc data/.gitignore
# git commit -m "inital setup"
# git status
# dvc status

# previous commit - 24d04d0
# changing the value from the original dataset  

# dvc add data/Telco-Customer-Churn.csv
# git add data/Telco-Customer-Churn.csv.dvc data/.gitignore

# latest commit - 59dd317

# git restore --source=24d04d0 data/Telco-Customer-Churn.csv.dvc
# dvc checkout

# git remote add origin https://github.com/mdshafwan/cusotmer_churn_dataset.git
# git remote -v
# git fetch origin
# git merge origin/main --allow-unrelated-histories

In [3]:
from pathlib import Path
import yaml

dcv_path = Path('data/Telco-Customer-Churn.csv.dvc')
metadata = yaml.safe_load(dcv_path.read_text())
dataset_version = metadata['outs'][0]['md5']
print('dataset_version',dataset_version)

dataset_version 836fd67bbd5c4bad12630810a4cd0f48


In [4]:
import mlflow
import mlflow.sklearn
import optuna
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from xgboost import XGBClassifier
from mlflow.models import infer_signature

data = pd.read_csv("data/Telco-Customer-Churn.csv")

data.drop("customerID",axis=1,inplace=True)

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"],errors="coerce")

x = data.drop("Churn",axis=1)
y = data["Churn"].map({"No":0,"Yes":1})

num = x.select_dtypes(include=["int64","float64"]).columns.tolist()
cat = x.select_dtypes(include=["object"]).columns.tolist()

numerical_pipeline = Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])

categorical_pipeline = Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore",drop="first"))])

preprocessing = ColumnTransformer([("num",numerical_pipeline,num),("cat",categorical_pipeline,cat)])

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

input_example = x_train.head()

mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("customer_churn_v2")

def objective(trial):

    params = {
        "max_depth":trial.suggest_int("max_depth",3,6),
        "n_estimators":trial.suggest_int("n_estimators",50,200),
        "learning_rate":trial.suggest_float("learning_rate",0.01,0.3,log=True),
        "subsample":trial.suggest_float("subsample",0.6,1.0),
        "colsample_bytree":trial.suggest_float("colsample_bytree",0.6,1.0)
    }

    with mlflow.start_run(nested=True):

        model = Pipeline([("preprocess",preprocessing),("model",XGBClassifier(**params))])

        model.fit(x_train,y_train)

        pred = model.predict(x_test)

        acc = accuracy_score(y_test,pred)

        mlflow.log_params(params)
        mlflow.log_param("dataset","customer_churn")
        mlflow.log_metric("accuracy",acc)

        return acc

with mlflow.start_run():

    study = optuna.create_study(direction="maximize")

    study.optimize(objective,n_trials=50)

    best_model = Pipeline([("preprocess",preprocessing),("model",XGBClassifier(**study.best_params))])

    best_model.fit(x_train,y_train)

    signature = infer_signature(x_train,best_model.predict(x_train))

    mlflow.sklearn.log_model(best_model,artifact_path="customer_model",input_example=input_example,signature=signature)

    mlflow.log_params(study.best_params)

    mlflow.log_metric("best_accuracy",study.best_value)

    mlflow.log_param('dataset_file','data/Telco-Customer-Churn.csv.')
    mlflow.log_param('dataset_version',dataset_version)
    final_pred = best_model.predict(x_test)
    final_accuracy = accuracy_score(y_test, final_pred)
    mlflow.log_metric("final_model_accuracy", final_accuracy)
    print(study.best_params)
    print(study.best_value)

[I 2026-10-28 20:33:22,986] A new study created in memory with name: no-name-8b98b4d0-7c62-4b14-8be9-cf396ceb3bc9
[I 2026-10-28 20:33:23,786] Trial 0 finished with value: 0.8005677785663591 and parameters: {'max_depth': 3, 'n_estimators': 156, 'learning_rate': 0.01673868653340342, 'subsample': 0.8184301866004092, 'colsample_bytree': 0.6770520301430005}. Best is trial 0 with value: 0.8005677785663591.


🏃 View run beautiful-mouse-626 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/b7b23dfefcf1411e84d8569de6fc0027
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


[I 2026-10-28 20:33:24,619] Trial 1 finished with value: 0.8161816891412349 and parameters: {'max_depth': 4, 'n_estimators': 172, 'learning_rate': 0.028038257304228084, 'subsample': 0.8112356538103739, 'colsample_bytree': 0.8135885728335874}. Best is trial 1 with value: 0.8161816891412349.


🏃 View run unequaled-conch-677 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/25f1469accd1477fbd7305dcc9137315
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


[I 2026-10-28 20:33:25,371] Trial 2 finished with value: 0.8161816891412349 and parameters: {'max_depth': 6, 'n_estimators': 54, 'learning_rate': 0.09352241732541687, 'subsample': 0.717256097449587, 'colsample_bytree': 0.6252412178532495}. Best is trial 1 with value: 0.8161816891412349.


...

🏃 View run marvelous-ram-484 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/f51050c5893b411b8bfca0a8172fac66
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


[I 2026-10-28 20:34:04,001] Trial 49 finished with value: 0.8105039034776437 and parameters: {'max_depth': 3, 'n_estimators': 73, 'learning_rate': 0.128325508590013, 'subsample': 0.6270757915095212, 'colsample_bytree': 0.8705834194326777}. Best is trial 22 with value: 0.8211497515968772.


🏃 View run polite-auk-97 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/63a0938e372f47069123b71df5b23672
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971


/mnt/d/brototype/SAFAN/code/fourtysix/mlops/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/28 20:34:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'max_depth': 3, 'n_estimators': 86, 'learning_rate': 0.12269845829424789, 'subsample': 0.611350800328685, 'colsample_bytree': 0.8777309931313468}
0.8211497515968772
🏃 View run nebulous-sheep-718 at: http://127.0.0.1:5000/#/experiments/907854656528108971/runs/170c523bfe7147fd9ac88263f2b0a8f9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/907854656528108971
